# Pipeline 產出物檢視

手動檢視 pipeline 執行後的產出物（parquet、pickle、json），用於除錯和驗證資料品質。

In [2]:
"""設定環境：載入配置與建立 DataCatalog"""
import os
from pathlib import Path
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()))

from recsys_tfb.core.config import ConfigLoader
from recsys_tfb.core.catalog import DataCatalog
from recsys_tfb.core.versioning import resolve_dataset_version

config = ConfigLoader("conf", env="local")
data_dir = Path("data")

# 解析版本
dataset_version = resolve_dataset_version(data_dir / "dataset", None)
model_version = "best"

try:
    params_inf = config.get_parameters_by_name("parameters_inference")
    inf_config = params_inf.get("inference", params_inf)
    snap_dates = inf_config.get("snap_dates", [])
    snap_date = snap_dates[0].replace("-", "") if snap_dates else "unknown"
except KeyError:
    snap_date = "unknown"

runtime_params = {
    "dataset_version": dataset_version,
    "model_version": model_version,
    "snap_date": snap_date,
}

catalog = DataCatalog(config.get_catalog_config(runtime_params=runtime_params))

print(f"Dataset version: {dataset_version}")
print(f"Model version: {model_version}")
print(f"Snap date: {snap_date}")
print("\n已註冊的資料集：")
for name in catalog.list():
    status = "存在" if catalog.exists(name) else "不存在"
    print(f"  - {name} ({status})")

Dataset version: 071ff0fe
Model version: best
Snap date: 20240331

已註冊的資料集：
  - feature_table (存在)
  - label_table (存在)
  - sample_keys (存在)
  - train_keys (存在)
  - train_dev_keys (存在)
  - val_keys (存在)
  - train_set (存在)
  - train_dev_set (存在)
  - val_set (存在)
  - X_train (存在)
  - y_train (存在)
  - X_train_dev (存在)
  - y_train_dev (存在)
  - X_val (存在)
  - y_val (存在)
  - model (存在)
  - best_params (存在)
  - evaluation_results (存在)
  - preprocessor (存在)
  - category_mappings (存在)
  - scoring_dataset (存在)
  - ranked_predictions (存在)


## Source Data

In [3]:
"""檢視 feature_table（Parquet）"""
if catalog.exists("feature_table"):
    df = catalog.load("feature_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())
else:
    print("feature_table 不存在，請先執行 dataset building pipeline。")

Shape: (600, 8)

Dtypes:
snap_date            datetime64[ns]
cust_id                      object
total_aum                   float64
fund_aum                    float64
in_amt_sum_l1m              float64
out_amt_sum_l1m             float64
in_amt_ratio_l1m            float64
out_amt_ratio_l1m           float64
dtype: object

Null 統計:
snap_date            0
cust_id              0
total_aum            0
fund_aum             0
in_amt_sum_l1m       0
out_amt_sum_l1m      0
in_amt_ratio_l1m     0
out_amt_ratio_l1m    0
dtype: int64


,snap_date,cust_id,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
0,2024-01-31,C000001,1202104.30,300971.57,40446.55,168057.34,0.033646,0.139803
1,2024-01-31,C000002,1168094.83,84042.96,13239.03,31761.59,0.011334,0.027191
2,2024-01-31,C000003,1192380.50,8308.68,21470.19,58477.36,0.018006,0.049043
3,2024-01-31,C000004,139897.14,16064.11,70948.52,35576.96,0.507148,0.254308
4,2024-01-31,C000005,43218.70,2848.59,878.64,29483.60,0.020330,0.682195
5,2024-01-31,C000006,726330.26,246102.00,88826.99,23012.49,0.122296,0.031683
6,2024-01-31,C000007,704980.35,42944.76,1057.21,975.01,0.001500,0.001383
7,2024-01-31,C000008,1562147.98,395481.14,47877.42,157156.66,0.030648,0.100603
8,2024-01-31,C000009,39647.10,13762.75,2952.64,41669.29,0.074473,1.051005
9,2024-01-31,C000010,523280.42,152043.47,16002.50,110398.02,0.030581,0.210973


,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
count,6.000000e+02,6.000000e+02,600.000000,600.000000,600.000000,600.000000
mean,5.187228e+05,1.251505e+05,50096.957517,42286.227150,0.440432,0.377019
std,5.342478e+05,1.606115e+05,51677.400056,42658.819182,1.937659,1.639452
min,6.904800e+02,8.389000e+01,24.210000,44.190000,0.000103,0.000196
25%,1.597485e+05,2.283696e+04,14033.107500,11118.692500,0.031773,0.026942
50%,3.615551e+05,7.010671e+04,32513.165000,28804.505000,0.096098,0.079920
75%,6.962938e+05,1.643068e+05,69245.105000,57599.990000,0.304552,0.219749
max,3.807333e+06,1.528173e+06,475905.990000,295499.890000,29.359678,26.377488


In [4]:
"""檢視 label_table（Parquet）"""
if catalog.exists("label_table"):
    df = catalog.load("label_table")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))
    display(df.describe())

    # Label 分佈
    if "label" in df.columns:
        print(f"\nLabel 分佈:\n{df['label'].value_counts()}")
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
        # 各產品正樣本比例
        pos_rate = df.groupby("prod_name")["label"].mean().sort_values(ascending=False)
        print(f"\n各產品正樣本比例:\n{pos_rate}")
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
else:
    print("label_table 不存在，請先執行 dataset building pipeline。")

Shape: (3000, 7)

Dtypes:
snap_date           datetime64[ns]
cust_id                     object
cust_segment_typ            object
apply_start_date    datetime64[ns]
apply_end_date      datetime64[ns]
label                        int64
prod_name                   object
dtype: object

Null 統計:
snap_date           0
cust_id             0
cust_segment_typ    0
apply_start_date    0
apply_end_date      0
label               0
prod_name           0
dtype: int64


,snap_date,cust_id,cust_segment_typ,apply_start_date,apply_end_date,label,prod_name
0,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,fx
1,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,usd
2,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,stock
3,2024-01-31,C000001,mass,2024-02-01,2024-03-01,0,bond
4,2024-01-31,C000001,mass,2024-02-01,2024-03-01,1,mix
5,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,fx
6,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,usd
7,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,stock
8,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,bond
9,2024-01-31,C000002,affluent,2024-02-01,2024-03-01,0,mix


,label
count,3000.000000
mean,0.099000
std,0.298712
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000



Label 分佈:
0    2703
1     297
Name: label, dtype: int64

Prod_name 分佈:
fx       600
usd      600
stock    600
bond     600
mix      600
Name: prod_name, dtype: int64

各產品正樣本比例:
prod_name
mix      0.111667
usd      0.108333
bond     0.096667
fx       0.093333
stock    0.085000
Name: label, dtype: float64

Snap_date 分佈:
2024-01-31    1000
2024-02-29    1000
2024-03-31    1000
Name: snap_date, dtype: int64


## Dataset Pipeline

In [5]:
"""載入 train_set、train_dev_set、val_set 並顯示 shape"""
import pandas as pd

splits = {}
for name in ["train_set", "train_dev_set", "val_set"]:
    if catalog.exists(name):
        splits[name] = catalog.load(name)
        print(f"{name}: {splits[name].shape}")
    else:
        print(f"{name} 不存在，請先執行 dataset pipeline。")

train_set: (1000, 13)
train_dev_set: (1000, 13)
val_set: (1000, 13)


In [6]:
"""基本統計比較：數值特徵 describe() — splits vs source feature_table"""
num_cols = ["total_aum", "fund_aum", "in_amt_sum_l1m", "out_amt_sum_l1m",
            "in_amt_ratio_l1m", "out_amt_ratio_l1m"]

source_ft = catalog.load("feature_table")
desc_parts = {"source_feature_table": source_ft[num_cols].describe()}
for name, df in splits.items():
    cols = [c for c in num_cols if c in df.columns]
    if cols:
        desc_parts[name] = df[cols].describe()

combined = pd.concat(desc_parts, axis=1)
display(combined)

source_feature_table                                               \
                 total_aum      fund_aum in_amt_sum_l1m out_amt_sum_l1m   
count         6.000000e+02  6.000000e+02     600.000000      600.000000   
mean          5.187228e+05  1.251505e+05   50096.957517    42286.227150   
std           5.342478e+05  1.606115e+05   51677.400056    42658.819182   
min           6.904800e+02  8.389000e+01      24.210000       44.190000   
25%           1.597485e+05  2.283696e+04   14033.107500    11118.692500   
50%           3.615551e+05  7.010671e+04   32513.165000    28804.505000   
75%           6.962938e+05  1.643068e+05   69245.105000    57599.990000   
max           3.807333e+06  1.528173e+06  475905.990000   295499.890000   

                                             train_set                 \
      in_amt_ratio_l1m out_amt_ratio_l1m     total_aum       fund_aum   
count       600.000000        600.000000  1.000000e+03    1000.000000   
mean          0.440432          0.377019  4.820590e+05  116105.152000   
std           1.937659          1.639452  4.560500e+05  133974.254952   
min           0.000103          0.000196  1.056988e+04     615.630000   
25%           0.031773          0.026942  1.506319e+05   23756.332500   
50%           0.096098          0.079920  3.424508e+05   67741.660000   
75%           0.304552          0.219749  6.727405e+05  162236.532500   
max          29.359678         26.377488  2.715795e+06  815702.440000   

                                      ...  train_dev_set                  \
      in_amt_sum_l1m out_amt_sum_l1m  ... in_amt_sum_l1m out_amt_sum_l1m   
count    1000.000000     1000.000000  ...     1000.00000     1000.000000   
mean    52004.138900    41593.088000  ...    50507.41235    41579.343850   
std     52018.834925    45902.520366  ...    48519.04987    38236.219839   
min       446.720000       44.190000  ...       24.21000      296.400000   
25%     14812.642500     8084.807500  ...    15550.47000    14043.875000   
50%     35176.565000    25593.875000  ...    32500.40500    29905.090000   
75%     71532.277500    58167.790000  ...    70027.95000    54156.002500   
max    308454.980000   295499.890000  ...   273001.95000   185943.870000   

                                               val_set                \
      in_amt_ratio_l1m out_amt_ratio_l1m     total_aum      fund_aum   
count      1000.000000       1000.000000  1.000000e+03  1.000000e+03   
mean          0.505646          0.511013  5.671532e+05  1.374100e+05   
std           2.310576          2.331209  6.099567e+05  1.900210e+05   
min           0.000103          0.001103  6.904800e+02  1.443900e+02   
25%           0.030865          0.030682  1.789510e+05  2.782388e+04   
50%           0.114644          0.070691  3.766059e+05  7.782327e+04   
75%           0.330313          0.203840  7.013086e+05  1.650297e+05   
max          24.789282         26.377488  3.807333e+06  1.528173e+06   

                                                                         
      in_amt_sum_l1m out_amt_sum_l1m in_amt_ratio_l1m out_amt_ratio_l1m  
count    1000.000000     1000.000000      1000.000000       1000.000000  
mean    47779.321300    43686.249600         0.488002          0.382629  
std     54194.748367    43401.478704         2.311383          1.536173  
min       592.620000      200.080000         0.001158          0.000473  
25%     12685.602500    10557.595000         0.031279          0.024870  
50%     30582.125000    30782.290000         0.088331          0.084846  
75%     64875.822500    62308.840000         0.234482          0.216884  
max    475905.990000   223432.220000        29.359678         18.925135  

[8 rows x 24 columns]

In [7]:
"""Label 分佈比較：各 split 正樣本率 vs source label_table"""
source_lt = catalog.load("label_table")

# Overall 正樣本率
overall_rates = {"source_label_table": source_lt["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns:
        overall_rates[name] = df["label"].mean()
print("Overall 正樣本率：")
for k, v in overall_rates.items():
    print(f"  {k}: {v:.4f}")

# Per-product 正樣本率
print("\nPer-product 正樣本率：")
per_prod_parts = {"source_label_table": source_lt.groupby("prod_name")["label"].mean()}
for name, df in splits.items():
    if "label" in df.columns and "prod_name" in df.columns:
        per_prod_parts[name] = df.groupby("prod_name")["label"].mean()

per_prod = pd.DataFrame(per_prod_parts)
display(per_prod)

Overall 正樣本率：
  source_label_table: 0.0990
  train_set: 0.1140
  train_dev_set: 0.0950
  val_set: 0.0880

Per-product 正樣本率：


,source_label_table,train_set,train_dev_set,val_set
prod_name,,,,
bond,0.096667,0.105,0.100,0.085
fx,0.093333,0.105,0.085,0.090
mix,0.111667,0.105,0.125,0.105
stock,0.085000,0.115,0.070,0.070
usd,0.108333,0.140,0.095,0.090


In [8]:
"""缺值比較：各 split null count vs source data"""
# Source data null counts
source_ft = catalog.load("feature_table")
source_lt = catalog.load("label_table")
source_nulls = pd.concat([source_ft.isnull().sum(), source_lt.isnull().sum()]).groupby(level=0).first()
source_nulls.name = "source_data"

# Splits null counts
null_parts = {"source_data": source_nulls}
for name, df in splits.items():
    null_parts[name] = df.isnull().sum()

null_df = pd.DataFrame(null_parts).fillna("-")
print("Null count 比較：")
display(null_df)

Null count 比較：


,source_data,train_set,train_dev_set,val_set
apply_end_date,0,0,0,0
apply_start_date,0,0,0,0
cust_id,0,0,0,0
cust_segment_typ,0,0,0,0
fund_aum,0,0,0,0
in_amt_ratio_l1m,0,0,0,0
in_amt_sum_l1m,0,0,0,0
label,0,0,0,0
out_amt_ratio_l1m,0,0,0,0
out_amt_sum_l1m,0,0,0,0


## Training Pipeline

In [9]:
"""檢視 category_mappings（JSON）"""
if catalog.exists("category_mappings"):
    mappings = catalog.load("category_mappings")
    print(f"Type: {type(mappings)}")
    print(f"\n內容：")
    print(mappings)
else:
    print("category_mappings 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

內容：
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}


In [10]:
"""檢視 preprocessor（Pickle）"""
if catalog.exists("preprocessor"):
    preprocessor = catalog.load("preprocessor")
    print(f"Type: {type(preprocessor)}")
    if isinstance(preprocessor, dict):
        print(f"\nKeys: {list(preprocessor.keys())}")
        for k, v in preprocessor.items():
            print(f"\n--- {k} ---")
            print(f"Type: {type(v)}")
            print(v)
    else:
        if hasattr(preprocessor, "get_params"):
            print(f"\nParams:\n{preprocessor.get_params()}")

else:
    print("preprocessor 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

Keys: ['feature_columns', 'categorical_columns', 'category_mappings', 'drop_columns']

--- feature_columns ---
Type: <class 'list'>
['prod_name', 'total_aum', 'fund_aum', 'in_amt_sum_l1m', 'out_amt_sum_l1m', 'in_amt_ratio_l1m', 'out_amt_ratio_l1m']

--- categorical_columns ---
Type: <class 'list'>
['prod_name']

--- category_mappings ---
Type: <class 'dict'>
{'prod_name': ['bond', 'fx', 'mix', 'stock', 'usd']}

--- drop_columns ---
Type: <class 'list'>
['snap_date', 'cust_id', 'label', 'apply_start_date', 'apply_end_date', 'cust_segment_typ']


In [11]:
"""檢視 model（Pickle）"""
if catalog.exists("model"):
    model = catalog.load("model")
    print(f"Type: {type(model)}")
    if hasattr(model, "get_params"):
        print(f"\nParams:\n{model.get_params()}")

    # LightGBM feature importance
    if hasattr(model, "feature_importances_") and hasattr(model, "feature_name_"):
        import pandas as pd
        importance = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_,
        }).sort_values("importance", ascending=False)
        print(f"\nTop 20 Feature Importance:")
        display(importance.head(20))
else:
    print("model 不存在，請先執行 training pipeline。")

Type: <class 'lightgbm.basic.Booster'>


In [12]:
"""檢視 best_params（JSON）：Optuna 搜出的最佳超參數"""
if catalog.exists("best_params"):
    import json
    best_params = catalog.load("best_params")
    print(f"Type: {type(best_params)}")
    print(f"\n最佳超參數：")
    print(json.dumps(best_params, indent=2, ensure_ascii=False))
else:
    print("best_params 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

最佳超參數：
{
  "learning_rate": 0.07206796917155728,
  "num_leaves": 50,
  "max_depth": 3,
  "min_child_samples": 89,
  "subsample": 0.87937861683409,
  "colsample_bytree": 0.7776788594864499
}


In [13]:
"""檢視 evaluation_results（JSON）：模型評估結果（mAP、per-product AP 等）"""
if catalog.exists("evaluation_results"):
    import json
    eval_results = catalog.load("evaluation_results")
    print(f"Type: {type(eval_results)}")
    print(f"\n評估結果：")
    print(json.dumps(eval_results, indent=2, ensure_ascii=False))
else:
    print("evaluation_results 不存在，請先執行 training pipeline。")

Type: <class 'dict'>

評估結果：
{
  "overall_map": 0.4827160493827161,
  "per_product_ap": {
    "bond": 0.09413754926691581,
    "fx": 0.08746168668060286,
    "mix": 0.12236111679790153,
    "stock": 0.1443124766697356,
    "usd": 0.09172137345561115
  },
  "n_queries": 200,
  "n_excluded_queries": 128
}


## Inference Pipeline

In [14]:
"""檢視 scoring_dataset（Parquet）：推論用的特徵資料集"""
if catalog.exists("scoring_dataset"):
    df = catalog.load("scoring_dataset")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # snap_date 分佈
    if "snap_date" in df.columns:
        print(f"\nSnap_date 分佈:\n{df['snap_date'].value_counts().sort_index()}")
    # prod_name 分佈
    if "prod_name" in df.columns:
        print(f"\nProd_name 分佈:\n{df['prod_name'].value_counts()}")
else:
    print("scoring_dataset 不存在，請先執行 inference pipeline。")

Shape: (1000, 9)

Dtypes:
snap_date            datetime64[ns]
cust_id                      object
prod_name                    object
total_aum                   float64
fund_aum                    float64
in_amt_sum_l1m              float64
out_amt_sum_l1m             float64
in_amt_ratio_l1m            float64
out_amt_ratio_l1m           float64
dtype: object

Null 統計:
snap_date            0
cust_id              0
prod_name            0
total_aum            0
fund_aum             0
in_amt_sum_l1m       0
out_amt_sum_l1m      0
in_amt_ratio_l1m     0
out_amt_ratio_l1m    0
dtype: int64


,snap_date,cust_id,prod_name,total_aum,fund_aum,in_amt_sum_l1m,out_amt_sum_l1m,in_amt_ratio_l1m,out_amt_ratio_l1m
0,2024-03-31,C000001,bond,176432.31,88112.82,56583.48,5241.63,0.320709,0.029709
1,2024-03-31,C000001,fx,176432.31,88112.82,56583.48,5241.63,0.320709,0.029709
2,2024-03-31,C000001,mix,176432.31,88112.82,56583.48,5241.63,0.320709,0.029709
3,2024-03-31,C000001,stock,176432.31,88112.82,56583.48,5241.63,0.320709,0.029709
4,2024-03-31,C000001,usd,176432.31,88112.82,56583.48,5241.63,0.320709,0.029709
5,2024-03-31,C000002,bond,14359.38,1607.86,98144.04,5710.31,6.834839,0.397671
6,2024-03-31,C000002,fx,14359.38,1607.86,98144.04,5710.31,6.834839,0.397671
7,2024-03-31,C000002,mix,14359.38,1607.86,98144.04,5710.31,6.834839,0.397671
8,2024-03-31,C000002,stock,14359.38,1607.86,98144.04,5710.31,6.834839,0.397671
9,2024-03-31,C000002,usd,14359.38,1607.86,98144.04,5710.31,6.834839,0.397671



Snap_date 分佈:
2024-03-31    1000
Name: snap_date, dtype: int64

Prod_name 分佈:
bond     200
fx       200
mix      200
stock    200
usd      200
Name: prod_name, dtype: int64


In [15]:
"""檢視 ranked_predictions（Parquet）：排序後的推薦結果"""
if catalog.exists("ranked_predictions"):
    df = catalog.load("ranked_predictions")
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull 統計:\n{df.isnull().sum()}")
    display(df.head(10))

    # 分數分佈統計
    if "score" in df.columns:
        print(f"\n分數分佈：")
        display(df["score"].describe())

    # 各產品平均分數
    if "prod_name" in df.columns and "score" in df.columns:
        avg_score = df.groupby("prod_name")["score"].mean().sort_values(ascending=False)
        print(f"\n各產品平均分數:\n{avg_score}")

    # 排名分佈：確認每位客戶有完整 1~N 排名
    if "rank" in df.columns and "cust_id" in df.columns:
        ranks_per_cust = df.groupby("cust_id")["rank"].agg(["min", "max", "count"])
        print(f"\n每位客戶排名統計：")
        display(ranks_per_cust.describe())
else:
    print("ranked_predictions 不存在，請先執行 inference pipeline。")

Shape: (1000, 5)

Dtypes:
snap_date    datetime64[ns]
cust_id              object
prod_code            object
score               float64
rank                  int64
dtype: object

Null 統計:
snap_date    0
cust_id      0
prod_code    0
score        0
rank         0
dtype: int64


,snap_date,cust_id,prod_code,score,rank
0,2024-03-31,C000001,bond,0.122171,1
1,2024-03-31,C000001,fx,0.122171,2
2,2024-03-31,C000001,mix,0.122171,3
3,2024-03-31,C000001,stock,0.122171,4
4,2024-03-31,C000001,usd,0.122171,5
5,2024-03-31,C000002,bond,0.122171,1
6,2024-03-31,C000002,fx,0.122171,2
7,2024-03-31,C000002,mix,0.122171,3
8,2024-03-31,C000002,stock,0.122171,4
9,2024-03-31,C000002,usd,0.122171,5



分數分佈：


count    1000.000000
mean        0.113742
std         0.003579
min         0.107551
25%         0.111323
50%         0.113931
75%         0.115401
max         0.122171
Name: score, dtype: float64


每位客戶排名統計：


,min,max,count
count,200.0,200.0,200.0
mean,1.0,5.0,5.0
std,0.0,0.0,0.0
min,1.0,5.0,5.0
25%,1.0,5.0,5.0
50%,1.0,5.0,5.0
75%,1.0,5.0,5.0
max,1.0,5.0,5.0
